In [ ]:
# Importing the libraries
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from scipy.stats import spearmanr
from scipy.stats import chi2_contingency, fisher_exact
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import files

uploaded = files.upload()

# ── Load data ──────────────────────────────────────────────────────────────────
df = pd.read_csv('medical_data_clean.csv')

# Identify numeric feature columns (exclude metadata)
meta_cols = ['Age', 'Sex', 'Smoking', 'LC_stage', 'LC_type', 'Group', 'Group_name', 'lscm']
feature_cols = [c for c in df.columns if c not in meta_cols]

# Split groups
lc      = df[df['Group_name'] == 'LC group'][feature_cols]
healthy = df[df['Group_name'] == 'Healthy group'][feature_cols]

print(f"LC group:      {len(lc)} samples")
print(f"Healthy group: {len(healthy)} samples")
print(f"Features tested: {len(feature_cols)}\n")

Saving medical_data_clean.csv to medical_data_clean (2).csv
LC group:      65 samples
Healthy group: 53 samples
Features tested: 18



In [ ]:
#Mann-Whitney U test (Sensors vs Group)
results = []
for col in feature_cols:
    lc_vals  = lc[col].dropna()
    hc_vals  = healthy[col].dropna()
    stat, p  = mannwhitneyu(lc_vals, hc_vals, alternative='two-sided')

    # Effect size: rank-biserial correlation
    n1, n2   = len(lc_vals), len(hc_vals)
    r        = 1 - (2 * stat) / (n1 * n2)

    results.append({
        'Feature'     : col,
        'p_value'     : p,
        'effect_size_r': r,
        'LC_median'   : np.median(lc_vals),
        'HC_median'   : np.median(hc_vals),
        'LC_mean'     : np.mean(lc_vals),
        'HC_mean'     : np.mean(hc_vals),
    })

results_df = pd.DataFrame(results).sort_values('p_value').reset_index(drop=True)
display(results_df.head())

,Feature,p_value,effect_size_r,LC_median,HC_median,LC_mean,HC_mean
0,S4_T2,1.144629e-15,0.859797,0.000004,0.000016,0.000006,0.000017
1,S4_T1,1.964049e-14,0.821480,0.000003,0.000010,0.000004,0.000011
2,S4_T3,9.919518e-14,0.798839,0.000006,0.000020,0.000008,0.000024
3,S3_T3,3.746972e-08,0.590711,0.000042,0.000100,0.000056,0.000121
4,S6_T3,6.772130e-08,0.579390,0.000001,0.000002,0.000001,0.000002


In [ ]:
#FDR correction
def fdr_bh(pvals, alpha=0.05):
    """Benjamini-Hochberg FDR correction."""
    n = len(pvals)
    order = np.argsort(pvals)
    ranked_p = np.array(pvals)[order]
    # adjusted p-values
    adj = np.minimum(1, ranked_p * n / (np.arange(1, n+1)))
    # enforce monotonicity from right
    for i in range(n-2, -1, -1):
        adj[i] = min(adj[i], adj[i+1])
    np.empty(n)[order] = adj
    return np.empty(n) < alpha, np.empty(n)

reject, p_fdr = fdr_bh(results_df['p_value'].values)
results_df['p_fdr']     = p_fdr
results_df['significant'] = reject   # FDR < 0.05

sig_df = results_df[results_df['significant']]
print(f"Significant features after FDR correction (q<0.05): {len(sig_df)}")
print(sig_df[['Feature','p_value','p_fdr','effect_size_r',
              'LC_median','HC_median']].to_string(index=False))

Significant features after FDR correction (q<0.05): 9
Feature      p_value        p_fdr  effect_size_r    LC_median    HC_median
  S4_T2 1.144629e-15 2.060333e-14       0.859797 4.300000e-06 1.550000e-05
  S4_T1 1.964049e-14 1.767645e-13       0.821480 2.850000e-06 1.010000e-05
  S4_T3 9.919518e-14 5.951711e-13       0.798839 6.440000e-06 2.010000e-05
  S3_T3 3.746972e-08 1.686137e-07       0.590711 4.210000e-05 1.003450e-04
  S6_T3 6.772130e-08 2.437967e-07       0.579390 1.190000e-06 2.170000e-06
  S6_T2 1.975207e-06 5.925620e-06       0.510595 1.120000e-06 1.700000e-06
  S3_T2 3.449268e-06 8.869547e-06       0.498403 9.790000e-05 2.083600e-04
  S3_T1 2.361097e-03 5.312469e-03       0.326560 2.334350e-04 3.587730e-04
  S1_T3 7.166219e-03 1.433244e-02       0.288824 3.100000e-07 4.860000e-07


Spearman Correlation Test

In [ ]:
#Spearman-correlation analysis
# Encode Group as binary: LC=1, Healthy=0
df['Group_bin'] = (df['Group_name'] == 'LC group').astype(int)

target_vars = {
    'Group'   : ('Group_bin',  'LC vs Healthy (1=LC, 0=Healthy)'),
    'Age'     : ('Age',        'Age (years)'),
    'Sex'     : ('Sex',        'Sex (binary)'),
    'Smoking' : ('Smoking',    'Smoking status (binary)'),
}



In [ ]:
all_results = {}

for label, (col, description) in target_vars.items():
    print(f"\n{'─'*60}")
    print(f"  Target: {label} — {description}")
    print(f"{'─'*60}")

    rows = []
    for feat in feature_cols:
        valid = df[[feat, col]].dropna()
        rho, p = spearmanr(valid[feat], valid[col])
        rows.append({'Feature': feat, 'rho': rho, 'p_value': p, 'n': len(valid)})

    res = pd.DataFrame(rows).sort_values('p_value').reset_index(drop=True)
    reject, p_fdr = fdr_bh(res['p_value'].values)
    res['p_fdr']       = p_fdr
    res['significant'] = reject
    res['target']      = label

    sig = res[res['significant']]
    print(f"  Significant (FDR<0.05): {len(sig)} / {len(feature_cols)} features")
    if len(sig):
        print(res[res['significant']][['Feature', 'rho', 'p_value', 'p_fdr']].to_string(index=False))

    all_results[label] = res


────────────────────────────────────────────────────────────
  Target: Group — LC vs Healthy (1=LC, 0=Healthy)
────────────────────────────────────────────────────────────
  Significant (FDR<0.05): 9 / 18 features
Feature       rho      p_value        p_fdr
  S4_T2 -0.740799 8.921287e-22 1.605832e-20
  S4_T1 -0.707767 3.224658e-19 2.902192e-18
  S4_T3 -0.688260 7.218457e-18 4.331074e-17
  S3_T3 -0.508945 3.987789e-09 1.794505e-08
  S6_T3 -0.499213 8.708761e-09 3.135154e-08
  S6_T2 -0.439937 6.206495e-07 1.861949e-06
  S3_T2 -0.429410 1.221229e-06 3.140302e-06
  S3_T1 -0.281353 2.026358e-03 4.559306e-03
  S1_T3 -0.248850 6.581445e-03 1.316289e-02

────────────────────────────────────────────────────────────
  Target: Age — Age (years)
────────────────────────────────────────────────────────────
  Significant (FDR<0.05): 10 / 18 features
Feature       rho  p_value    p_fdr
  S4_T2 -0.328967 0.000276 0.004960
  S4_T3 -0.307978 0.000692 0.006228
  S3_T2 -0.283643 0.001855 0.011129
  S3_T3

In [ ]:
#Mann-Whitney U test (Age Vs Group)
lc_age = df[df['Group_name'] == 'LC group']['Age'].dropna()
hc_age = df[df['Group_name'] == 'Healthy group']['Age'].dropna()

stat, p = mannwhitneyu(lc_age, hc_age, alternative='two-sided')
n1, n2  = len(lc_age), len(hc_age)
r_effect = 1 - (2 * stat) / (n1 * n2)

print(f"\n{'─'*60}")
print(f"  Age in LC Group     : Mean={lc_age.mean():.1f}  Median={lc_age.median():.1f}  SD={lc_age.std():.1f}")
print(f"  Age in Healthy Group: Mean={hc_age.mean():.1f}  Median={hc_age.median():.1f}  SD={hc_age.std():.1f}")
print(f"{'─'*60}")
print(f"  U statistic   : {stat:.1f}")
print(f"  p-value       : {p:.6f}  ({'*** SIGNIFICANT' if p < 0.05 else 'Not significant'} at α=0.05)")
print(f"  Effect size r : {r_effect:.4f}  ({'Large' if abs(r_effect)>0.5 else 'Medium' if abs(r_effect)>0.3 else 'Small'})")



────────────────────────────────────────────────────────────
  Age in LC Group     : Mean=64.6  Median=66.0  SD=8.7
  Age in Healthy Group: Mean=56.0  Median=58.0  SD=12.3
────────────────────────────────────────────────────────────
  U statistic   : 2484.0
  p-value       : 0.000038  (*** SIGNIFICANT at α=0.05)
  Effect size r : -0.4421  (Medium)
────────────────────────────────────────────────────────────


In [ ]:
# ── Contingency table ─────────────────────────────────────────────────────────
ct = pd.crosstab(df['Group_name'], df['Smoking'],
                 rownames=['Group'], colnames=['Smoking'])
ct.columns = ['Non-Smoker (0)', 'Smoker (1)']
ct.index   = ['Healthy Group', 'LC Group']

print("=" * 65)
print("  CHI-SQUARE TEST: SMOKING vs GROUP")
print("=" * 65)
print("\nContingency Table:")
print(ct.to_string())

# Row & column totals
ct_vals = ct.values
n_total = ct_vals.sum()

# ── Chi-square test ───────────────────────────────────────────────────────────
chi2, p, dof, expected = chi2_contingency(ct_vals, correction=False)
chi2_yates, p_yates, _, _ = chi2_contingency(ct_vals, correction=True)   # Yates' correction

# ── Fisher's exact test (gold standard for 2x2) ───────────────────────────────
odds_ratio, p_fisher = fisher_exact(ct_vals)

# ── Effect sizes ──────────────────────────────────────────────────────────────
# Cramer's V
cramer_v = np.sqrt(chi2 / (n_total * (min(ct_vals.shape) - 1)))

# Phi coefficient (2x2 specific)
a, b = ct_vals[0]   # Healthy: non-smoker, smoker
c, d = ct_vals[1]   # LC:      non-smoker, smoker
phi = (a*d - b*c) / np.sqrt((a+b)*(c+d)*(a+c)*(b+d))

# Risk ratios & attributable risk
n_healthy = ct_vals[0].sum()
n_lc      = ct_vals[1].sum()
p_smoke_lc      = d / n_lc
p_smoke_healthy = b / n_healthy
relative_risk   = p_smoke_lc / p_smoke_healthy
attr_risk       = p_smoke_lc - p_smoke_healthy

print(f"\n{'─'*65}")
print(f"  Expected frequencies (under H₀):")
exp_df = pd.DataFrame(expected,
                      index=['Healthy Group','LC Group'],
                      columns=['Non-Smoker (0)','Smoker (1)'])
print(exp_df.round(2).to_string())

print(f"""
{'─'*65}
  TEST RESULTS
{'─'*65}
  Chi-square (no correction) : χ²({dof}) = {chi2:.4f},  p = {p:.6f}
  Chi-square (Yates corrected): χ²({dof}) = {chi2_yates:.4f},  p = {p_yates:.6f}
  Fisher's Exact Test         : OR = {odds_ratio:.4f},  p = {p_fisher:.6f}

  Effect sizes
  ────────────
  Cramér's V : {cramer_v:.4f}  ({'Large' if cramer_v>=0.5 else 'Medium' if cramer_v>=0.3 else 'Small'} effect)
  Phi (φ)    : {phi:.4f}

  Prevalence & Risk
  ─────────────────
  Smoking prevalence in LC group      : {p_smoke_lc*100:.1f}%  ({int(d)}/{n_lc})
  Smoking prevalence in Healthy group : {p_smoke_healthy*100:.1f}%  ({int(b)}/{n_healthy})
  Relative Risk (LC vs Healthy)       : {relative_risk:.3f}
  Attributable Risk                   : {attr_risk*100:.1f} percentage points
  Odds Ratio (Fisher)                 : {odds_ratio:.3f}
{'─'*65}""")


  CHI-SQUARE TEST: SMOKING vs GROUP

Contingency Table:
               Non-Smoker (0)  Smoker (1)
Healthy Group              40          13
LC Group                   22          43

─────────────────────────────────────────────────────────────────
  Expected frequencies (under H₀):
               Non-Smoker (0)  Smoker (1)
Healthy Group           27.85       25.15
LC Group                34.15       30.85

─────────────────────────────────────────────────────────────────
  TEST RESULTS
─────────────────────────────────────────────────────────────────
  Chi-square (no correction) : χ²(1) = 20.2867,  p = 0.000007
  Chi-square (Yates corrected): χ²(1) = 18.6517,  p = 0.000016
  Fisher's Exact Test         : OR = 6.0140,  p = 0.000008

  Effect sizes
  ────────────
  Cramér's V : 0.4146  (Medium effect)
  Phi (φ)    : 0.4146

  Prevalence & Risk
  ─────────────────
  Smoking prevalence in LC group      : 66.2%  (43/65)
  Smoking prevalence in Healthy group : 24.5%  (13/53)
  Relative Risk

In [ ]:
#Chi-square test (sex vs group)

# ── Contingency table ─────────────────────────────────────────────────────────
ct2 = pd.crosstab(df['Group_name'], df['Sex'],
                 rownames=['Group'], colnames=['Sex'])
ct2.columns = ['Female (0)', 'Male (1)']
ct2.index   = ['Healthy Group', 'LC Group']

print("=" * 65)
print("  CHI-SQUARE TEST: Sex vs GROUP")
print("=" * 65)
print("\nContingency Table:")
print(ct2.to_string())

# Row & column totals
ct_vals2 = ct2.values
n_total = ct_vals2.sum()

# ── Chi-square test ───────────────────────────────────────────────────────────
chi2, p, dof, expected = chi2_contingency(ct_vals2, correction=False)
chi2_yates, p_yates, _, _ = chi2_contingency(ct_vals2, correction=True)   # Yates' correction

# ── Fisher's exact test (gold standard for 2x2) ───────────────────────────────
odds_ratio, p_fisher = fisher_exact(ct_vals2)

# ── Effect sizes ──────────────────────────────────────────────────────────────
# Cramer's V
cramer_v = np.sqrt(chi2 / (n_total * (min(ct_vals2.shape) - 1)))

# Phi coefficient (2x2 specific)
a, b = ct_vals2[0]   # Healthy: non-smoker, smoker
c, d = ct_vals2[1]   # LC:      non-smoker, smoker
phi = (a*d - b*c) / np.sqrt((a+b)*(c+d)*(a+c)*(b+d))

# Risk ratios & attributable risk
n_healthy = ct_vals2[0].sum()
n_lc      = ct_vals2[1].sum()
p_smoke_lc      = d / n_lc
p_smoke_healthy = b / n_healthy
relative_risk   = p_smoke_lc / p_smoke_healthy
attr_risk       = p_smoke_lc - p_smoke_healthy

print(f"\n{'─'*65}")
print(f"  Expected frequencies (under H₀):")
exp_df = pd.DataFrame(expected,
                      index=['Healthy Group','LC Group'],
                      columns=['Female (0)','Male (1)'])
print(exp_df.round(2).to_string())

print(f"""
{'─'*65}
  TEST RESULTS
{'─'*65}
  Chi-square (no correction) : χ²({dof}) = {chi2:.4f},  p = {p:.6f}
  Chi-square (Yates corrected): χ²({dof}) = {chi2_yates:.4f},  p = {p_yates:.6f}
  Fisher's Exact Test         : OR = {odds_ratio:.4f},  p = {p_fisher:.6f}

  Effect sizes
  ────────────
  Cramér's V : {cramer_v:.4f}  ({'Large' if cramer_v>=0.5 else 'Medium' if cramer_v>=0.3 else 'Small'} effect)
  Phi (φ)    : {phi:.4f}

  Prevalence & Risk
  ─────────────────
  Smoking prevalence in LC group      : {p_smoke_lc*100:.1f}%  ({int(d)}/{n_lc})
  Smoking prevalence in Healthy group : {p_smoke_healthy*100:.1f}%  ({int(b)}/{n_healthy})
  Relative Risk (LC vs Healthy)       : {relative_risk:.3f}
  Attributable Risk                   : {attr_risk*100:.1f} percentage points
  Odds Ratio (Fisher)                 : {odds_ratio:.3f}
{'─'*65}""")

  CHI-SQUARE TEST: Sex vs GROUP

Contingency Table:
               Female (0)  Male (1)
Healthy Group          26        27
LC Group               23        42

─────────────────────────────────────────────────────────────────
  Expected frequencies (under H₀):
               Female (0)  Male (1)
Healthy Group       22.01     30.99
LC Group            26.99     38.01

─────────────────────────────────────────────────────────────────
  TEST RESULTS
─────────────────────────────────────────────────────────────────
  Chi-square (no correction) : χ²(1) = 2.2474,  p = 0.133835
  Chi-square (Yates corrected): χ²(1) = 1.7197,  p = 0.189737
  Fisher's Exact Test         : OR = 1.7585,  p = 0.188420

  Effect sizes
  ────────────
  Cramér's V : 0.1380  (Small effect)
  Phi (φ)    : 0.1380

  Prevalence & Risk
  ─────────────────
  Smoking prevalence in LC group      : 64.6%  (42/65)
  Smoking prevalence in Healthy group : 50.9%  (27/53)
  Relative Risk (LC vs Healthy)       : 1.268
  Attributab